# LLM Conversational Memory & State Management

This notebook explores implementing stateful AI agents using **LangChain** and **LCEL**. It covers the transition from stateless LLM calls to persistent conversational interfaces using various memory strategies.

### Key Implementation Patterns:
* **Conversation Buffer Memory:** Storing raw message history for exact context retrieval.
* **Conversation Summary Memory:** Using an LLM to compress long transcripts into concise state.
* **LCEL Integration:** Utilizing `RunnableWithMessageHistory` for modular, chain-based memory.

---

In [1]:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

import os
from langchain_groq import ChatGroq

from dotenv import load_dotenv
load_dotenv()

# Initialize the Groq model cleanly
llama_llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.2,
    max_tokens=256
)

### Memory


Most LLM applications have a conversational interface. An essential component of a conversation is being able to refer to information introduced earlier in the conversation. At a bare minimum, a conversational system should be able to directly access some window of past messages.


In [2]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

In [3]:
# Define the Prompt (Using the dynamic array!)
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful, concise AI assistant."),
    # This placeholder dynamically injects the past conversation here
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

In [4]:
# Build the Raw LCEL Chain. Notice this is just a standard chain. It has no memory yet.
raw_chain = prompt | llama_llm

In [5]:
# The Database Strategy (Session Management)
# In production, this would be Redis or Postgres. Here, it is RAM.
session_store = {}

def get_session_history(session_id: str):
    """Fetches or creates a chat history for a specific user session."""
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
    return session_store[session_id]

In [6]:
# Wrap the Chain with the State Manager
conversational_chain = RunnableWithMessageHistory(
    runnable=raw_chain,
    get_session_history=get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

In [7]:
# Execution & Testing
# We must pass a config dict with a session_id to route the memory correctly.
config = {"configurable": {"session_id": "user_nabeel_123"}}

print("--- Turn 1 ---")
response1 = conversational_chain.invoke(
    {"input": "Hello, I am a little cat. Who are you?"}, 
    config=config
)
print(f"AI: {response1.content}\n")

--- Turn 1 ---
AI: Hello little kitty. I'm a friendly AI assistant, here to help and chat with you. I don't have a physical body, but I can understand and respond to your meows and questions. What would you like to talk about? Do you want to play, learn something new, or just have some fun?



In [8]:
print("--- Turn 2 ---")
response2 = conversational_chain.invoke(
    {"input": "What can you do?"}, 
    config=config
)
print(f"AI: {response2.content}\n")

--- Turn 2 ---
AI: I can do lots of things. Here are some of my favorite things:

1. **Answer questions**: I can help you learn about the world, from simple things like what's for dinner to more complex topics like science and history.
2. **Play games**: We can play text-based games like "Find the Treat" or "Whisper the Word".
3. **Tell stories**: I can make up fun stories just for you, or we can create a story together.
4. **Chat about your day**: If you want to talk about your adventures, I'm all ears (or rather, all text).
5. **Teach you new things**: I can help you learn new words, concepts, and skills.
6. **Generate fun facts**: I can share interesting and fun facts about the world.
7. **Create a virtual adventure**: We can go on a virtual adventure together, exploring new places and having exciting experiences.

What sounds like fun to you, little kitty?



In [9]:
print("--- Turn 3 (The Memory Test) ---")
response3 = conversational_chain.invoke(
    {"input": "Who am I?"}, 
    config=config
)
print(f"AI: {response3.content}")

--- Turn 3 (The Memory Test) ---
AI: You're a little cat, remember? You're a curious and playful feline friend who's here to explore and have fun with me.


In [10]:
session_store

{'user_nabeel_123': InMemoryChatMessageHistory(messages=[HumanMessage(content='Hello, I am a little cat. Who are you?', additional_kwargs={}, response_metadata={}), AIMessage(content="Hello little kitty. I'm a friendly AI assistant, here to help and chat with you. I don't have a physical body, but I can understand and respond to your meows and questions. What would you like to talk about? Do you want to play, learn something new, or just have some fun?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 56, 'total_tokens': 121, 'completion_time': 0.136182952, 'completion_tokens_details': None, 'prompt_time': 0.021188171, 'prompt_tokens_details': None, 'queue_time': 0.007434092, 'total_time': 0.157371123}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d04e4-63e9-72f0-8a2f-513f235fa1da-0', tool_ca

In [11]:
print("\n--- Conversation History ---")
history = get_session_history("user_nabeel_123").messages
for i, msg in enumerate(history):
    print(f"Message {i+1} - Role: {type(msg).__name__}, Content: {msg.content[:50]}...")


--- Conversation History ---
Message 1 - Role: HumanMessage, Content: Hello, I am a little cat. Who are you?...
Message 2 - Role: AIMessage, Content: Hello little kitty. I'm a friendly AI assistant, h...
Message 3 - Role: HumanMessage, Content: What can you do?...
Message 4 - Role: AIMessage, Content: I can do lots of things. Here are some of my favor...
Message 5 - Role: HumanMessage, Content: Who am I?...
Message 6 - Role: AIMessage, Content: You're a little cat, remember? You're a curious an...


In [12]:
print("\n" + "="*50)
print("INSPECTING THE MEMORY DATABASE")
print("="*50)

# Check the active sessions in our "Database"
print(f"Active User Sessions in RAM: {list(session_store.keys())}\n")

# Extract the specific memory object for user
target_user = "user_nabeel_123"
user_memory_object = session_store[target_user]

# Inspect the physical message array
raw_message_array = user_memory_object.messages

print(f"Total Messages Logged for {target_user}: {len(raw_message_array)}\n")

# Loop through and deconstruct the exact Python objects
for index, msg in enumerate(raw_message_array):
    # Determine the strict Object Class (HumanMessage vs AIMessage)
    msg_type = type(msg).__name__
    
    print(f"Index [{index}] | Class: <{msg_type}>")
    print(f"Payload: '{msg.content}'")
    print("-" * 40)


INSPECTING THE MEMORY DATABASE
Active User Sessions in RAM: ['user_nabeel_123']

Total Messages Logged for user_nabeel_123: 6

Index [0] | Class: <HumanMessage>
Payload: 'Hello, I am a little cat. Who are you?'
----------------------------------------
Index [1] | Class: <AIMessage>
Payload: 'Hello little kitty. I'm a friendly AI assistant, here to help and chat with you. I don't have a physical body, but I can understand and respond to your meows and questions. What would you like to talk about? Do you want to play, learn something new, or just have some fun?'
----------------------------------------
Index [2] | Class: <HumanMessage>
Payload: 'What can you do?'
----------------------------------------
Index [3] | Class: <AIMessage>
Payload: 'I can do lots of things. Here are some of my favorite things:

1. **Answer questions**: I can help you learn about the world, from simple things like what's for dinner to more complex topics like science and history.
2. **Play games**: We can play

In [13]:
from langchain_core.output_parsers import StrOutputParser

# Database Strategy (Session Management)
# We use a dictionary to track different user sessions.
session_store = {}

def get_session_history(session_id: str):
    """Fetches or creates a chat history for a specific user session."""
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
    return session_store[session_id]

# Build the Buffer Memory Chatbot
# Define the prompt with a dynamic memory injection slot
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Keep your answers short."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

# Create the raw LCEL execution graph
raw_chain = prompt | llama_llm | StrOutputParser()

# Wrap the chain with the State Manager
buffer_chatbot = RunnableWithMessageHistory(
    runnable=raw_chain,
    get_session_history=get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

In [14]:
# ---------------------------------------------------------
# Function to Simulate a Conversation
# ---------------------------------------------------------
def chat_simulation(chatbot, session_id, inputs):
    """Runs a series of inputs through the chatbot and displays responses."""
    print(f"\n=== Beginning Chat Simulation (Session: {session_id}) ===")
    config = {"configurable": {"session_id": session_id}}
    
    for i, user_input in enumerate(inputs):
        print(f"\n--- Turn {i+1} ---")
        print(f"Human: {user_input}")
        
        # Invoke the chatbot with the session configuration
        response = chatbot.invoke({"input": user_input}, config=config)
        print(f"AI: {response}")
    
    print("\n=== End of Chat Simulation ===")

# Execute the Buffer Memory Test
test_inputs = [
    "Hello, my name is Alice.",
    "My favorite color is blue.",
    "I enjoy hiking in the mountains.",
    "Can you remember both my name and my favorite color?"
]

In [15]:
chat_simulation(buffer_chatbot, "alice_buffer_session", test_inputs)


=== Beginning Chat Simulation (Session: alice_buffer_session) ===

--- Turn 1 ---
Human: Hello, my name is Alice.
AI: Hello Alice, nice to meet you. How can I assist you today?

--- Turn 2 ---
Human: My favorite color is blue.
AI: Blue is a calming color. Do you have a favorite shade of blue, like sky blue or navy?

--- Turn 3 ---
Human: I enjoy hiking in the mountains.
AI: That sounds beautiful. Are you a fan of summiting peaks or do you prefer more leisurely, scenic hikes?

--- Turn 4 ---
Human: Can you remember both my name and my favorite color?
AI: Your name is Alice, and your favorite color is blue.

=== End of Chat Simulation ===


In [26]:
# Examine the stored memory in RAM
print("\nFinal Buffer Memory Contents:")
final_memory = session_store["alice_buffer_session"].messages
for msg in final_memory:
    print(f"{type(msg).__name__}: {msg.content}")


Final Buffer Memory Contents:
HumanMessage: Hello, my name is Alice.
AIMessage: Hello Alice, nice to meet you. How can I assist you today?
HumanMessage: My favorite color is blue.
AIMessage: Blue is a calming color. Do you have a favorite shade of blue, like sky blue or navy?
HumanMessage: I enjoy hiking in the mountains.
AIMessage: That sounds beautiful. Are you a fan of summiting peaks or do you prefer more leisurely, scenic hikes?
HumanMessage: Can you remember both my name and my favorite color?
AIMessage: Your name is Alice, and your favorite color is blue.


In [30]:
# ---------------------------------------------------------
# Build a Custom Summary Memory Chatbot
# ---------------------------------------------------------

In [27]:
# Import the LangChain utility to flatten message objects into a string
from langchain_core.messages import get_buffer_string

In [28]:
summary_template = """You are an expert AI summarizer. Read the following chat transcript and summarize it in one sentence, focusing strictly on the user's personal preferences and name.

TRANSCRIPT:
{transcript}

SUMMARY:"""

summary_prompt = ChatPromptTemplate.from_template(summary_template)

# Assemble the LCEL Summarizer Chain
summarizer_chain = summary_prompt | llama_llm | StrOutputParser()

# Extract the raw messages and FLATTEN them into a single string
current_history = session_store["alice_buffer_session"].messages
transcript_string = get_buffer_string(current_history)
print("\n=== Chat Transcript ===")
print(transcript_string)


=== Chat Transcript ===
Human: Hello, my name is Alice.
AI: Hello Alice, nice to meet you. How can I assist you today?
Human: My favorite color is blue.
AI: Blue is a calming color. Do you have a favorite shade of blue, like sky blue or navy?
Human: I enjoy hiking in the mountains.
AI: That sounds beautiful. Are you a fan of summiting peaks or do you prefer more leisurely, scenic hikes?
Human: Can you remember both my name and my favorite color?
AI: Your name is Alice, and your favorite color is blue.


In [29]:
# Execute! We pass the string into the template.
summary = summarizer_chain.invoke({"transcript": transcript_string})

print("\nInstead of storing 10 raw messages, we store this compressed summary:")
print(f"Compressed State: {summary}")

print("\n=== Memory Comparison ===")
raw_char_count = sum(len(m.content) for m in current_history)
print(f"Buffer Memory Size (Raw): ~{raw_char_count} characters")
print(f"Summary Memory Size (Compressed): ~{len(summary)} characters")


Instead of storing 10 raw messages, we store this compressed summary:
Compressed State: Alice is a nature enthusiast who enjoys hiking in the mountains and has a personal preference for the color blue.

=== Memory Comparison ===
Buffer Memory Size (Raw): ~431 characters
Summary Memory Size (Compressed): ~113 characters
